# Data exploration and visualization

This notebook is dedicated to exploring the dataset and visualizing the features to gain insights into the data distribution and relationships between variables. We will use various plotting techniques to understand the characteristics of the dataset and identify any patterns or anomalies that may be present. 

The 3 main datasets wil be:
- **Top tagging**: This dataset contains information about top quark tagging, which is a technique used in particle physics to identify top quarks in high-energy collisions. The features in this dataset may include various kinematic variables and jet substructure observables that are relevant for top quark identification.
- **Quark-gluon jet tagging**: This dataset contains information about quark-gluon jet tagging, which is a technique used to distinguish between jets originating from quarks and gluons. The features in this dataset may include various jet substructure observables and kinematic variables that are relevant for quark-gluon discrimination.
- **Higgs tagging**: This dataset contains information about Higgs boson tagging, which is a technique used to identify Higgs bosons in high-energy collisions. The features in this dataset may include various kinematic variables and jet substructure observables that are relevant for Higgs boson identification.

## Top tagging 

We can find the dataset in the following path: `../data/val.h5`. We use the val test to not load 1 GB from train set. We will load the dataset and perform some exploratory data analysis to understand the distribution of the features and the relationships between them. We will start by loading the dataset and displaying some basic information about it, such as the number of samples, the number of features, and the distribution of the target variable. We will then create various plots to visualize the data, such as histograms, scatter plots, and correlation matrices. This will help us identify any patterns or anomalies in the data that may be relevant for our analysis.

**Description taken from the original source:** [zenodo](https://zenodo.org/records/2603256)

A set of MC simulated training/testing events for the evaluation of top quark tagging architectures.

In total 1.2M training events, 400k validation events and 400k test events. Use “train” for training, “val” for validation during the training and “test” for final testing and reporting results.

Description

- 14 TeV, hadronic tops for signal, qcd diets background, Delphes ATLAS detector card with Pythia8

- No MPI/pile-up included

- Clustering of  particle-flow entries (produced by Delphes E-flow) into anti-kT 0.8 jets in the pT range [550,650] GeV

- All top jets are matched to a parton-level top within ∆R = 0.8, and to all top decay partons within 0.8

- Jets are required to have |$\eta$| < 2

- The leading 200 jet constituent four-momenta are stored, with zero-padding for jets with fewer than 200

- Constituents are sorted by pT, with the highest pT one first

- The truth top four-momentum is stored as truth_px etc.

- A flag (1 for top, 0 for QCD) is kept for each jet. It is called is_signal_new

- The variable "ttv" (= test/train/validation) is kept for each jet. It indicates to which dataset the jet belongs. It is redundant as the different sets are already distributed as different files.

In [ ]:
# Cargamos las librerías necesarias
import h5py
import hdf5plugin

In [19]:
try:
    with h5py.File('../data/val.h5', 'r') as f:
        print("File keys:", list(f.keys()))
        table = f['table']
        print("Table keys:", list(table.keys()))
        
        print("\n_i_table keys:", list(table['_i_table'].keys())) 
        print("table columns:", list(table['table'].dtype.names))

except Exception as e:
    print(f"Error loading data using hdf5plugin: {e}")

File keys: ['table']
Table keys: ['_i_table', 'table']

_i_table keys: ['index']
table columns: ['index', 'values_block_0', 'values_block_1']


### _i_table key

In [30]:
try:
    with h5py.File('../data/val.h5', 'r') as f:
        table = f['table']['_i_table']['index']
        print("_i_table keys:", list(table.keys()), "\n")
        
        for key in table.keys():
            print(f"{key}: shape {table[key].shape}, dtype {table[key].dtype}")

except Exception as e:
    print(f"Error loading data using hdf5plugin: {e}")

_i_table keys: ['abounds', 'bounds', 'indices', 'indicesLR', 'mbounds', 'mranges', 'ranges', 'sorted', 'sortedLR', 'zbounds'] 

abounds: shape (384,), dtype int64
bounds: shape (3, 127), dtype int64
indices: shape (3, 131072), dtype uint32
indicesLR: shape (131072,), dtype uint32
mbounds: shape (384,), dtype int64
mranges: shape (3,), dtype int64
ranges: shape (3, 2), dtype int64
sorted: shape (3, 131072), dtype int64
sortedLR: shape (131201,), dtype int64
zbounds: shape (384,), dtype int64


This `_i_table` entry is a list of Optimized Index, so we will focus on `table` entry.

### table key

In [31]:
try:
    with h5py.File('../data/val.h5', 'r') as f:   
        table = f['table']['table']
        print("Table keys:", list(table.dtype.names), "\n")
        
        for key in table.dtype.names:
            print(f"{key}: shape {table[key].shape}, dtype {table[key].dtype}")
except Exception as e:
    print(f"Error loading data using hdf5plugin: {e}")

Table keys: ['index', 'values_block_0', 'values_block_1'] 

index: shape (403000,), dtype int64
values_block_0: shape (403000, 804), dtype float32
values_block_1: shape (403000, 2), dtype int64


Let's see what is inside

### Data exploration

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
# import h5py
# import hdf5plugin

In [35]:
try:
    with h5py.File('../data/val.h5', 'r') as f:
        table = f['table']['table']
        x_qt_data = pd.DataFrame(table['values_block_0'][:1000])
        y_qt_data = pd.DataFrame(table['values_block_1'][:1000])

        print("QG Data loaded successfully using hdf5plugin.")
except Exception as e:
    print(f"Error: {e}")

QG Data loaded successfully using hdf5plugin.


In [37]:
x_qt_data.head(10)

,0,1,2,3,4,5,6,7,8,9,...,794,795,796,797,798,799,800,801,802,803
0,266.676880,85.227501,-227.714478,109.539703,145.654129,46.856384,-124.277214,59.789429,76.116646,24.653267,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,385.807739,82.459038,303.009888,-224.127502,107.484184,15.370172,85.792786,-62.898380,69.148743,14.434706,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,144.535309,-108.092529,93.247643,-22.612755,70.963295,-21.669872,46.335262,-49.185867,56.287312,-19.737480,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,82.668480,49.805248,-9.679499,-65.267319,58.262596,38.636459,-7.670141,-42.929283,59.068157,37.110703,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,406.581726,186.811081,-345.960571,103.545181,42.170246,19.375856,-35.882679,10.739602,36.417843,12.717323,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,383.579987,-2.425069,331.046387,-193.742126,59.478207,-21.276777,52.788219,-17.273096,48.719555,-4.679730,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,145.315002,1.308301,130.312241,64.291985,106.912674,1.862807,97.060287,44.790062,34.371437,0.424950,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,257.993561,-56.336994,206.284210,144.338623,229.137405,-65.462891,180.365311,125.247421,127.783592,-27.591572,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,481.208313,-129.306366,105.096413,-451.437744,454.156525,-120.728462,100.163246,-426.204315,125.675858,-34.692989,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,87.898865,-4.744920,85.308998,-20.641462,55.467308,6.759329,53.698742,-12.139971,38.524090,-19.097017,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# We see all the information about the data
x_qt_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Columns: 804 entries, 0 to 803
dtypes: float32(804)
memory usage: 3.1 MB


## Quark-Gluon jets

The next dataset is Quark-Gluon. The dataset is also in `../data/QG_jets_fp32_0.npz`. In this case we use numpy to read the file.

In [3]:
import numpy as np

In [5]:
try:
    # we use numpy to read the file
    with np.load('../data/QG_jets_fp32_0.npz') as data:
        
        print("Data loaded successfully using numpy.")
        print("Data keys:", data.keys)
except Exception as e:
    print(f"Error loading data using numpy: {e}")

Data loaded successfully using numpy.
Data keys: <bound method NpzFile.keys of NpzFile '../data/QG_jets_fp32_0.npz' with keys: X, y>


## Higgs